# PLA2G2 preprocessing

Input data for Figure 6 

## Raw ProtSpace files
| File | Contents |
|------|----------|
| `Pla2g2.fasta` | Protein sequences |
| `Pla2g2.csv` | Metadata: `identifier`, `gene`, `group` (e.g. `C mammals`) |

## Derived feature columns
| Column | Derivation |
|--------|------------|
| `enzyme_class` | First token of `group` (e.g. `C`, `D1`) |
| `species` | Remaining tokens of `group` (e.g. `mammals`, `reptilia`) |
| `seq_length` | Length of amino acid sequence from FASTA |
| `length_bin` | Sequence length discretised into 6 equal-width bins |

In [ ]:
import os
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:


_cwd = Path.cwd()
REPO_ROOT = next((str(p) for p in [_cwd, *_cwd.parents] if (p / ".git").exists()), str(_cwd))

RAW_FASTA    = os.path.join(REPO_ROOT, "examples/paper/data/pla2g2/raw/Pla2g2.fasta")
RAW_METADATA = os.path.join(REPO_ROOT, "examples/paper/data/pla2g2/raw/Pla2g2.csv")
OUTPUT_FASTA = os.path.join(REPO_ROOT, "examples/paper/data/pla2g2/processed/Pla2g2.fasta")
OUTPUT_CSV   = os.path.join(REPO_ROOT, "examples/paper/data/pla2g2/processed/Pla2g2_features.csv")

## 1. Download raw data

In [ ]:
_BASE = "https://raw.githubusercontent.com/tsenoner/protspace/main/data/Pla2g2"

_DOWNLOADS = {
    RAW_FASTA:    f"{_BASE}/Pla2g2.fasta",
    RAW_METADATA: f"{_BASE}/Pla2g2.csv",
}

for dest, url in _DOWNLOADS.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"saved to {dest}")
    else:
        print(f"Already present: {dest}")

## 2. Parse sequences

In [ ]:
sequences = {}
with open(RAW_FASTA) as fh:
    cur_id, cur_seq = None, []
    for line in fh:
        line = line.rstrip()
        if line.startswith(">"):
            if cur_id is not None:
                sequences[cur_id] = "".join(cur_seq)
            cur_id, cur_seq = line[1:].split()[0], []
        else:
            cur_seq.append(line)
    if cur_id is not None:
        sequences[cur_id] = "".join(cur_seq)

print(f"Sequences in FASTA: {len(sequences)}")

## 3. Parse metadata and build features

The ProtSpace CSV has three columns: `identifier`, `gene`, `group`.
`enzyme_class` and `species` are derived by splitting `group` on the first space
(e.g. `"D1 reptilia"` → class=`"D1"`, species=`"reptilia"`).
Proteins with missing annotations or absent from the FASTA are dropped.

In [ ]:
KNOWN_CLASSES = {"A", "B", "C", "D", "D1", "D2", "D3", "E", "F", "G", "V"}

meta = pd.read_csv(RAW_METADATA)
print(f"Metadata rows: {len(meta)} | Columns: {list(meta.columns)}")

# Derive enzyme_class and species from the 'group' column
meta["enzyme_class"] = meta["group"].str.split().str[0]
meta["species"]      = meta["group"].str.split(n=1).str[1]

# Add sequence length; filter to FASTA proteins with complete and valid annotations
meta["seq_length"] = meta["identifier"].map(lambda i: len(sequences.get(i, "")))
features = meta.dropna(subset=["enzyme_class", "species"]).copy()
features = features[features["identifier"].isin(sequences)].copy()
features = features[features["enzyme_class"].isin(KNOWN_CLASSES)].reset_index(drop=True)
print(f"Proteins after filtering: {len(features)}")

# Bin sequence length into 6 equal-width bins
min_len, max_len = features["seq_length"].min(), features["seq_length"].max()
bins = np.linspace(min_len, max_len, num=7)
bin_labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins) - 1)]
features["length_bin"] = pd.cut(features["seq_length"], bins=bins, labels=bin_labels, include_lowest=True)

features.head()

## 4. Save outputs

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_FASTA), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

features.to_csv(OUTPUT_CSV, index=False)
print(f"Saved features to {OUTPUT_CSV}")

with open(OUTPUT_FASTA, "w") as fh:
    for pid in features["identifier"]:
        fh.write(f">{pid}\n{sequences[pid]}\n")
print(f"Saved {len(features)} sequences to {OUTPUT_FASTA}")